# Residual Stream Projection

Project the residual stream onto meaningful directions: token directions,
difference directions, PCA, embedding subspace, and multi-direction tracking.

## Why This Matters

Residual stream stream projection characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_projection import (
    project_to_token_direction, project_to_difference_direction,
    residual_pca_projection, project_to_embedding_subspace,
    multi_direction_projection,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Project to Token Direction

How strongly does the residual point toward a target token?

In [ ]:
result = project_to_token_direction(model, tokens, target_token=42)
print(f"Target token: {result['target_token']}")
print(f"Final projection: {result['final_projection']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: mean={p['mean_projection']:.4f}, "
          f"last_pos={p['last_position_projection']:.4f}")

## Difference Direction

Does the model favor token A or B at each layer?

In [ ]:
result = project_to_difference_direction(model, tokens, token_a=42, token_b=73)
print(f"Token A={result['token_a']}, Token B={result['token_b']}")
print(f"Final preference: {result['final_preference']}\n")
for p in result['per_layer']:
    fav = 'A' if p['favors_a'] else 'B'
    print(f"  Layer {p['layer']}: proj={p['projection']:+.4f} → favors {fav}")

## PCA Projection

What are the dominant directions of variation?

In [ ]:
result = residual_pca_projection(model, tokens, layer=2, n_components=3)
print(f"Total variance explained: {result['total_variance_explained']:.2%}\n")
for c in result['per_component']:
    print(f"  PC{c['component']}: var_expl={c['variance_explained']:.2%}, "
          f"sv={c['singular_value']:.4f}")
    print(f"    projections: {[f'{p:.3f}' for p in c['projections']]}")

## Embedding Subspace

How much stays in the original embedding space?

In [ ]:
result = project_to_embedding_subspace(model, tokens, layer=2)
moved = 'BEYOND' if result['moved_beyond_embed'] else 'within'
print(f"Mean in-embed: {result['mean_fraction_in_embed']:.2%} [{moved}]\n")
for p in result['per_position']:
    bar = '█' * int(p['fraction_in_embed'] * 20)
    print(f"  Pos {p['position']}: in_embed={p['fraction_in_embed']:.2%}, "
          f"orthogonal={p['fraction_orthogonal']:.2%} {bar}")

## Multi-Direction Projection

Track preference among multiple tokens through layers.

In [ ]:
result = multi_direction_projection(model, tokens, target_tokens=[10, 42, 73, 90])
print(f"Final winner: token {result['final_winner']}\n")
for p in result['per_layer']:
    projs = ', '.join(f'{t}:{v:+.3f}' for t, v in sorted(p['projections'].items()))
    print(f"  Layer {p['layer']}: winner={p['winner']} ({projs})")